# Lab 04 — Hybrid Search & Reranking from Scratch

**Pairs with:** [Course 06 · RAG](https://psssnikhil.github.io/ai-engineering-handbook/build/module-09-rag-retrieval-augmented-generation/) and [Course 10 · Vector DBs](https://psssnikhil.github.io/ai-engineering-handbook/build/module-13-vector-databases-deep-dive/)

In this lab, you will build a production-grade **Hybrid Retrieval & Reranking Engine** from scratch in pure Python with **no frameworks**.

You will implement:
1. **Sparse Lexical Search (BM25)** — Term frequency, inverse document frequency, and length normalization.
2. **Dense Vector Search (Cosine Similarity)** — Dense embeddings with dot-product similarity.
3. **Reciprocal Rank Fusion (RRF)** — Combining sparse + dense rankings into a single robust candidate list.
4. **Cross-Encoder Reranking** — Scoring top-k candidate chunks with a cross-encoder model to maximize Precision@k.

```
Query ──► [BM25 Sparse Search]  ──┐
        │                         ├─► Reciprocal Rank Fusion (RRF) ─► Cross-Encoder Reranker ─► Top Chunks
        └──► [Dense Vector Search] ──┘
```

**Prerequisites:** `pip install -r requirements.txt` (scikit-learn, numpy, sentence-transformers or sklearn).

In [ ]:
import math
import numpy as np
from collections import Counter
from typing import List, Dict, Any, Tuple
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

## Step 1: Sample Document Corpus

Let's create a realistic technical corpus with domain jargon where pure vector search often struggles with exact code symbols or keywords.

In [ ]:
CORPUS = [
    {"id": "doc_1", "text": "vLLM uses PagedAttention to reduce KV-cache memory fragmentation in GPU VRAM."},
    {"id": "doc_2", "text": "LiteLLM provides a unified API wrapper for calling OpenAI, Anthropic, Bedrock, and Cohere."},
    {"id": "doc_3", "text": "Grouped Query Attention (GQA) reduces KV-cache memory overhead by sharing key-value heads across query heads."},
    {"id": "doc_4", "text": "Ragas evaluates Retrieval Augmented Generation pipelines using faithfulness and answer relevance metrics."},
    {"id": "doc_5", "text": "PagedAttention divides continuous KV-cache into fixed-size virtual pages to maximize batch concurrency."}
]

## Step 2: Implement BM25 (Sparse Lexical Search)

BM25 calculates relevance scores based on exact term matches, inverse document frequency (IDF), and document length normalization.

In [ ]:
class MinimalBM25:
    def __init__(self, corpus: List[str], k1: float = 1.5, b: float = 0.75):
        self.k1 = k1
        self.b = b
        self.corpus = [doc.lower().split() for doc in corpus]
        self.doc_len = [len(doc) for doc in self.corpus]
        self.avg_doc_len = sum(self.doc_len) / len(self.corpus)
        self.doc_freqs = Counter(word for doc in self.corpus for word in set(doc))
        self.N = len(corpus)
        
    def search(self, query: str) -> List[Tuple[int, float]]:
        q_tokens = query.lower().split()
        scores = []
        
        for idx, doc in enumerate(self.corpus):
            score = 0.0
            doc_len = self.doc_len[idx]
            tf_counter = Counter(doc)
            
            for token in q_tokens:
                if token in tf_counter:
                    df = self.doc_freqs[token]
                    idf = math.log((self.N - df + 0.5) / (df + 0.5) + 1.0)
                    tf = tf_counter[token]
                    denom = tf + self.k1 * (1.0 - self.b + self.b * (doc_len / self.avg_doc_len))
                    score += idf * (tf * (self.k1 + 1.0)) / denom
                    
            scores.append((idx, score))
            
        return sorted(scores, key=lambda x: x[1], reverse=True)

## Step 3: Implement Dense Vector Search (TF-IDF / Cosine Similarity)

Dense search measures semantic vector closeness regardless of exact term overlaps.

In [ ]:
def dense_search(query: str, corpus_texts: List[str]) -> List[Tuple[int, float]]:
    vectorizer = TfidfVectorizer().fit(corpus_texts + [query])
    corpus_vecs = vectorizer.transform(corpus_texts)
    query_vec = vectorizer.transform([query])
    sims = cosine_similarity(query_vec, corpus_vecs)[0]
    
    ranked = sorted(enumerate(sims), key=lambda x: x[1], reverse=True)
    return ranked

## Step 4: Reciprocal Rank Fusion (RRF)

RRF combines rankings from different retrieval algorithms without requiring normalized scores.

$$\text{RRF Score}(d) = \sum_{m \in M} \frac{1}{k + r_m(d)}$$

In [ ]:
def reciprocal_rank_fusion(rankings_list: List[List[Tuple[int, float]]], k: int = 60) -> List[Tuple[int, float]]:
    rrf_scores = {}
    
    for ranking in rankings_list:
        for rank, (doc_idx, _) in enumerate(ranking, start=1):
            if doc_idx not in rrf_scores:
                rrf_scores[doc_idx] = 0.0
            rrf_scores[doc_idx] += 1.0 / (k + rank)
            
    return sorted(rrf_scores.items(), key=lambda x: x[1], reverse=True)

## Step 5: Test Hybrid Search Pipeline

Let's test querying for `PagedAttention KV-cache`.

In [ ]:
query = "PagedAttention KV-cache GPU"
texts = [doc["text"] for doc in CORPUS]

bm25_model = MinimalBM25(texts)
bm25_ranks = bm25_model.search(query)
dense_ranks = dense_search(query, texts)

hybrid_results = reciprocal_rank_fusion([bm25_ranks, dense_ranks])

print(f"=== HYBRID RRF SEARCH RESULTS FOR QUERY: '{query}' ===\n")
for rank, (doc_idx, score) in enumerate(hybrid_results, start=1):
    doc = CORPUS[doc_idx]
    print(f"Rank {rank} | RRF Score: {score:.5f} | Doc ID: {doc['id']}")
    print(f"   Text: {doc['text']}\n")